WHO > [The Global Health Observatory: Explore a world of health data](https://www.who.int/data/gho/data/themes/topics/indicator-groups/indicator-group-details/GHO/life-expectancy-and-healthy-life-expectancy)<br />
WHO > [Life expectancy at birth](https://www.who.int/data/gho/data/indicators/indicator-details/GHO/life-expectancy-at-birth-(years))<br />
WHO > [Healthy life expectancy (HALE) at birth](https://www.who.int/data/gho/data/indicators/indicator-details/GHO/gho-ghe-hale-healthy-life-expectancy-at-birth)<br />

"[List of countries by life expectancy](https://en.wikipedia.org/wiki/List_of_countries_by_life_expectancy)"<br />
[Category:Life expectancy charts_by_country](https://commons.wikimedia.org/wiki/Category:Life_expectancy_charts_by_country)<br />
[Таблица подбора цветов](http://mal-bioit.ru/survey-web-colors)

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from collections import namedtuple
import math
import os

In [3]:
pd.options.display.max_rows = 100
pd.options.display.min_rows = 6
pd.options.display.max_columns = 50
pd.options.display.float_format = '{:.2f}'.format

CHART_COMPLEXITY = 'simple'  # 'simple', 'all_to_files'
                             # Options 'all_to_files' allows to create all varieties of charts simultaneously
DESTINATION_OUTPUT = 'pass'  # to where table code should be placed: 'file', 'show', 'pass'
VER_LINES=[]
LANG = 'en'

In [4]:
# list of countries that should be ignored during processing (it is needed than WBG made correction for only some countries.
# So processing of countries without correction is redundant.
LS_IGNORE = \
    []

LS_COUNTRIES_EXAMPLES = ['Russia', 'USA', 'France', 'Spain', 'Japan', 'Switzerland', 'South Korea', 'world']

In [5]:
dd_countries_renaming = {
	'Bolivia (Plurinational State of)': 'Bolivia',
	'Brunei Darussalam': 'Brunei',
	'Cabo Verde': 'Cape Verde',
	"Democratic People's Republic of Korea": 'North Korea',
	'Democratic Republic of the Congo': 'Congo, DR',
    'Congo': 'Congo Republic',
	'Iran (Islamic Republic of)': 'Iran',
	"Lao People's Democratic Republic": 'Laos',
	'Micronesia (Federated States of)': 'Micronesia',
	'Netherlands (Kingdom of the)': 'Netherlands',
	'Republic of Korea': 'South Korea',
	'Republic of Moldova': 'Moldova',
	'Russian Federation': 'Russia',
	'Syrian Arab Republic': 'Syria',
    'Türkiye': 'Turkey',
	'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
	'United Republic of Tanzania': 'Tanzania',
	'United States of America': 'USA',
	'Venezuela (Bolivarian Republic of)': 'Venezuela',
    'Viet Nam': 'Vietnam',
	'occupied Palestinian territory, including east Jerusalem': 'occupied Palestinian territory',
    'Global': 'world'
}

In [6]:
def load_data_from_csv(file_name_core, selected_indicator):

    def load_single_csv(file_name):
        return pd.read_csv(f"data/{file_name}", usecols=['Indicator', 'Location', 'Period', 'Dim1', 'FactValueNumeric'])
        
    def get_pretty_df_for_sex(df, selected_sex):
        df_sex = df[df.sex == selected_sex] \
                   .drop(columns='sex') \
                   .pivot(index='country', columns='year', values='value')
        df_sex.index.name = ''
        df_sex.columns.name = ''
        return df_sex


    df = pd.concat([load_single_csv(f"{file_name_core} -countries.csv"),
                    load_single_csv(f"{file_name_core} -global.csv"),
                    load_single_csv(f"{file_name_core} -regions.csv"),
                    load_single_csv(f"{file_name_core} -income_groups.csv")])
    
    df.rename(columns={'Location': 'country',
                       'Period': 'year',
                       'Dim1': 'sex',
                       'FactValueNumeric': 'value'}, inplace=True)
    df = df[df.Indicator == selected_indicator] \
              .drop(columns='Indicator')
    df['country'] = df['country'].replace(dd_countries_renaming)

    df_total = get_pretty_df_for_sex(df, 'Both sexes')
    df_male = get_pretty_df_for_sex(df, 'Male')
    df_female = get_pretty_df_for_sex(df, 'Female')

    return df_total, df_male, df_female



(df_le_total, df_le_male, df_le_female) = load_data_from_csv(file_name_core="Life expectancy at birth",
                                                             selected_indicator="Life expectancy at birth (years)")
df_le_total.loc[LS_COUNTRIES_EXAMPLES].sort_values(by=2019, ascending=False)

,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
,,,,,,,,,,,,,,,,,,,,,,
Japan,81.53,81.84,82.10,82.14,82.37,82.24,82.59,82.76,82.83,83.16,83.08,82.83,83.33,83.56,83.79,84.00,84.16,84.26,84.35,84.47,84.66,84.46
South Korea,75.86,76.57,76.94,77.44,77.93,78.37,78.97,79.38,79.91,80.36,80.54,80.95,81.13,81.69,82.16,82.40,82.73,83.09,83.15,83.69,83.80,83.80
Switzerland,79.71,80.10,80.41,80.40,80.94,81.11,81.41,81.60,81.86,81.89,82.20,82.37,82.39,82.48,82.85,82.59,83.17,83.14,83.29,83.48,82.74,83.33
Spain,79.09,79.39,79.49,79.41,80.00,80.01,80.67,80.68,80.97,81.33,81.69,81.81,81.96,82.48,82.58,82.33,82.73,82.68,82.77,83.14,81.92,82.66
France,78.91,79.01,79.14,79.09,80.00,80.00,80.53,80.80,80.87,81.04,81.22,81.58,81.58,81.82,82.25,81.91,82.13,82.17,82.38,82.53,81.87,81.92
USA,76.66,76.75,76.85,76.99,77.40,77.38,77.68,77.94,78.05,78.42,78.61,78.65,78.77,78.76,78.80,78.61,78.56,78.50,78.63,78.74,76.89,76.37
Russia,65.16,65.10,64.86,64.72,65.14,65.19,66.49,67.42,67.79,68.56,68.83,69.77,70.19,70.70,70.69,71.30,71.62,72.50,72.66,73.22,71.38,70.02
world,66.77,67.10,67.36,67.62,67.95,68.39,68.98,69.42,69.70,70.22,70.50,70.93,71.31,71.66,71.95,72.19,72.44,72.68,72.89,73.12,72.45,71.37


In [7]:
(df_hale_total, df_hale_male, df_hale_female) = load_data_from_csv(file_name_core="Healthy life expectancy (HALE) at birth",
                                                                   selected_indicator="Healthy life expectancy (HALE) at birth (years)")
df_hale_total.loc[LS_COUNTRIES_EXAMPLES].sort_values(by=2019, ascending=False)

,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
,,,,,,,,,,,,,,,,,,,,,,
Japan,71.11,71.34,71.53,71.57,71.74,71.66,71.92,72.04,72.10,72.34,72.31,72.12,72.54,72.74,72.93,73.10,73.24,73.37,73.48,73.58,73.64,73.40
South Korea,66.55,67.10,67.40,67.77,68.14,68.47,68.93,69.23,69.63,69.96,70.11,70.42,70.56,70.94,71.27,71.45,71.71,72.03,72.12,72.50,72.50,72.45
Spain,68.89,69.12,69.19,69.13,69.53,69.53,69.99,69.98,70.18,70.43,70.69,70.77,70.88,71.22,71.28,71.11,71.38,71.35,71.42,71.69,70.63,71.11
Switzerland,68.35,68.67,68.97,69.06,69.50,69.67,69.87,69.99,70.15,70.13,70.36,70.50,70.56,70.64,70.92,70.76,71.20,71.22,71.36,71.53,70.87,71.15
France,67.98,68.04,68.14,68.12,68.75,68.75,69.12,69.33,69.42,69.55,69.70,69.97,70.00,70.20,70.52,70.24,70.41,70.51,70.61,70.72,70.08,70.08
USA,65.32,65.36,65.43,65.53,65.82,65.81,66.00,66.16,66.22,66.44,66.56,66.56,66.59,66.53,66.50,66.31,66.17,66.03,66.02,66.02,64.43,63.91
Russia,56.71,56.69,56.56,56.51,56.87,56.95,58.00,58.77,59.11,59.75,59.99,60.76,61.12,61.57,61.63,62.15,62.43,63.15,63.29,63.72,62.22,60.94
world,58.12,58.40,58.63,58.87,59.15,59.52,60.01,60.39,60.64,61.08,61.32,61.68,61.99,62.27,62.51,62.71,62.91,63.10,63.27,63.45,62.83,61.91


<br>
<br>

In [9]:
ChartParams = namedtuple('ChartParams', ['padding_down', 'padding_up', 'add_empty_labels_down', 'add_empty_labels_up',
                                         'legend_loc', 'legend_ncol', 'label_x_rotation'],
                                         defaults=(0, 0, 0, 0, 'upper left', 3, False))

In [10]:
def rounding_up(x):
    rem = x % 1
    return (math.ceil(x) + 1)  if rem == 0 else \
           (math.ceil(x))  if rem <= 0.25 else \
           (math.ceil(x) + 0.5) if rem <= 0.75 else \
           (math.ceil(x) + 1)


def rounding_down(x):
    rem = x % 1
    return (math.floor(x) - 1)  if rem == 0 else \
           (math.floor(x))  if rem >= 0.75 else \
           (math.floor(x) - 0.5) if rem >= 0.2 else \
           (math.floor(x) - 1)       

In [11]:
# Ensure that all directories for output of files exist. Otherwise, create them.
def ensure_existance_of_directories(ls_dir):
    for dir in ls_dir:
        if not os.path.exists(dir):
            os.mkdir(dir)
    print('Existance of directories checked')
            
ls_dir = ['output-charts-Africa', 'output-charts-America', 'output-charts-Asia', 
          'output-charts-Europe', 'output-charts-groups', 'output-charts-new']
ensure_existance_of_directories(ls_dir)

Existance of directories checked


In [12]:
def process_chart_result(plt, lang=LANG, destination=DESTINATION_OUTPUT, chart_complexity=CHART_COMPLEXITY, file_name=None):   
    if destination == 'file':
        clarification_of_folder = '-Europe' if file_name in [
            'Albania', 'Austria', 'Belarus', 'Belgium', 'Bosnia and Herzegovina', 'Bulgaria', 'Croatia', 'Czechia', 'Denmark', 'Estonia',
            'Faroe Islands', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Latvia', 'Liechtenstein',
            'Lithuania', 'Luxembourg', 'Malta', 'Moldova', 'Montenegro', 'Netherlands', 'North Macedonia', 'Norway', 'Poland', 'Portugal',
            'Romania', 'Russia', 'Serbia', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Ukraine', 'United Kingdom'
        ] else '-Asia' if file_name in [
            'Afghanistan', 'Armenia', 'Australia', 'Azerbaijan', 'Bangladesh', 'Bhutan', 'Cambodia', 'China', 'Cyprus', 'French Polynesia',
            'Georgia', 'Hong Kong SAR', 'India', 'Indonesia', 'Iran', 'Iraq', 'Israel', 'Japan', 'Jordan', 'Kazakhstan', 'Kuwait',
            'Kyrgyzstan', 'Lebanon', 'Macao SAR', 'Malaysia', 'Maldives', 'Mongolia', 'Myanmar', 'Nepal', 'New Caledonia', 'New Zealand',
            'North Korea', 'Oman', 'Pakistan', 'Philippines', 'Qatar', 'Saudi Arabia', 'Singapore', 'South Korea', 'Sri Lanka', 'Syria',
            'Tajikistan', 'Thailand', 'Turkey', 'Turkmenistan', 'United Arab Emirates', 'Uzbekistan', 'Vietnam', 'West Bank and Gaza', 'Yemen'
        ] else '-Africa' if file_name in [
            'Algeria', 'Angola', 'Botswana', 'Burkina Faso', 'Cameroon', 'Cape Verde', 'Central African Republic', 'Chad', 'Congo Republic',
            'Congo, DR', "Cote d'Ivoire", 'Egypt', 'Eswatini', 'Ethiopia', 'Ghana', 'Kenya', 'Lesotho', 'Liberia', 'Libya', 'Madagascar',
            'Malawi', 'Mali', 'Mauritania', 'Mauritius', 'Morocco', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 'Rwanda', 'Senegal',
            'Seychelles', 'Somalia', 'South Africa', 'Sudan', 'Tanzania', 'Togo', 'Tunisia', 'Uganda', 'Zambia', 'Zimbabwe'
        ] else '-America' if file_name in [
            'Argentina', 'Bermuda', 'Bolivia', 'Brazil', 'British Virgin Islands', 'Canada', 'Chile', 'Colombia', 'Costa Rica', 'Cuba',
            'Dominican Republic', 'Ecuador', 'El Salvador', 'Guatemala', 'Haiti', 'Honduras', 'Jamaica', 'Mexico', 'Nicaragua', 'Panama',
            'Paraguay', 'Peru', 'Puerto Rico', 'Saint-Martin (France)', 'Trinidad and Tobago', 'US Virgin Islands', 'USA', 'Uruguay', 'Venezuela'
        ] else '-new' if file_name in [
        ] else '-new'
        plt.savefig(f"output-charts{clarification_of_folder}/HALE and Life Expectancy by WHO -{file_name}{' -ru' if lang=='ru' else ''}.png",
                        bbox_inches='tight', facecolor='white', pad_inches=0.05)        
        print('Data has written to file')
    else:
        plt.show()

In [13]:
def create_chart_simple(ser_le_total=None, ser_le_male=None, ser_le_female=None,
                        ser_hale_total=None, ser_hale_male=None, ser_hale_female=None, *,
                        figure_size=(15, 10), title_en='', title_ru='', ver_lines=[], 
                        lang='en', chart_params=ChartParams(), file_name=''):

    plt.rcParams['figure.figsize'] = figure_size

    min_chart = min(ser_hale_total.min(), ser_hale_male.min(), ser_hale_female.min())
    max_chart = max(ser_le_total.max(), ser_le_male.max(), ser_le_female.max())
    print(f"min: {min_chart:.2f}, max: {max_chart:.2f}               Δ{(max_chart - min_chart):.2f}")

    # ratio HALE / LE in 2000, 2019, 2021
    ratio_2000 = 100 * ser_hale_total.loc[2000] / ser_le_total.loc[2000]
    ratio_2019 = 100 * ser_hale_total.loc[2019] / ser_le_total.loc[2019]
    ratio_2021 = 100 * ser_hale_total.loc[2021] / ser_le_total.loc[2021]
    print(f"{ratio_2000:.2f}  Δ{(ratio_2019-ratio_2000):.2f}  {ratio_2019:.2f}  Δ{(ratio_2021-ratio_2019):.2f}  {ratio_2021:.2f}" +
          f"   -ratio HALE / LE, %")

    # # alternative ration: means for 2000-02, 2017-19, 2020-21
    # mean_2000_02 = 100 * ser_hale_total.loc[2000:2002].mean() / ser_le_total.loc[2000:2002].mean()
    # mean_2017_19 = 100 * ser_hale_total.loc[2017:2019].mean() / ser_le_total.loc[2017:2019].mean()
    # mean_2020_21 = 100 * ser_hale_total.loc[2020:2021].mean() / ser_le_total.loc[2020:2021].mean()
    # print(f"{mean_2000_02:.2f}  Δ{(mean_2017_19-mean_2000_02):.2f}  {mean_2017_19:.2f}  Δ{(mean_2020_21-mean_2017_19):.2f}  {mean_2020_21:.2f}" +
    #       f"   -ratio HALE / LE, %")
          

    # determine limits on axis
    ylim = (rounding_down(min_chart) - chart_params.padding_down,
            rounding_up(max_chart)   + chart_params.padding_up)

    min_for_labels = math.ceil(ylim[0])
    max_for_labels = math.floor(ylim[1])
    # print(f"min_for_labels / max_for_labels: {min_for_labels} - {max_for_labels}")

    # determine which label have to be not shown
    empty_labels_down = (round(min_chart) - min_for_labels) + chart_params.add_empty_labels_down
    empty_labels_up   = max_for_labels - round(max_chart) + chart_params.add_empty_labels_up

    # determine range of years
    year_min = ser_le_total.index[0]
    year_max = ser_le_total.index[-1]

    # set chart areas, their locations, sizes and appearances
    fig = plt.figure()
    ax = fig.add_subplot(111)

    
    # set where ticks and labels will be shown around charts
    ax.tick_params(axis='x', which='both', top=False, bottom=True, labeltop=False, labelbottom=True, pad=1.5)
    ax.tick_params(axis='y', which='both', left=True, right=True, labelleft=True, labelright=True, pad=1.5)

    # set limits of chart axes
    ax.set_xlim(year_min, year_max)
    ax.set_ylim(ylim[0], ylim[1])


    # set ticks and labels
    labels_x = range(year_min, year_max+1)
    ax.set_xticks(labels_x)
    ax.set_xticklabels(labels_x, fontsize=12.5, rotation=chart_params.label_x_rotation)
    
    labels_y = tuple(range(min_for_labels, max_for_labels + 1))
    ax.set_yticks(labels_y)
    ax.set_yticklabels(('',) * empty_labels_down + 
                        labels_y[empty_labels_down:-empty_labels_up if empty_labels_up else None] +
                        ('',) * empty_labels_up, fontsize=11.5)

    # settings grids
    ax.grid(color='wheat', linewidth=0.25)

    for year in range(year_min - year_min % 10 + 10, year_max, 10):
        ax.axvline(x=year, color='lightgrey', linewidth=0.4, zorder=0)

    for age in range(min_for_labels - min_for_labels % 5 + 5, max_for_labels, 5):
        ax.axhline(y=age, color='lightgrey', linewidth=0.4, zorder=0)

    for year in ver_lines:
        ax.axvline(x=year, color='wheat', linewidth=2, alpha=0.5, zorder=0)

    # set title
    ax.set_title(label=f'Ожидаемая продолжительность жизни и ожидаемая продолжительность здоровой жизни {title_ru}' if lang=='ru' else \
                        f'Life Expectancy and Healthy Life Expectancy (HALE) {title_en}', fontsize=14)

    if chart_params.legend_ncol==3:
        ax.plot(ser_le_female.index, ser_le_female.values, color='lightpink', linestyle='-', linewidth=3, label='женщины, ОПЖ' if lang=='ru' else 'female, LE', zorder=12)
        ax.plot(ser_hale_female.index, ser_hale_female.values, color='red', linestyle='-', linewidth=3, label='женщины, ОПЗЖ' if lang=='ru' else 'female, HALE', zorder=15)    
        ax.plot(ser_le_total.index, ser_le_total.values, color='palegreen', linestyle='-', linewidth=6, label='всё население, ОПЖ' if lang=='ru' else 'overall, LE', zorder=11)
        # ax.plot(ser_le_total.index, ser_le_total.values, color='greenyellow', linestyle='-', linewidth=6, label='всё население, ОПЖ' if lang=='ru' else 'overall, LE', zorder=11)
        ax.plot(ser_hale_total.index, ser_hale_total.values, color='forestgreen', linestyle='-', linewidth=6, label='всё население, ОПЗЖ' if lang=='ru' else 'overall, HALE', zorder=14)
        ax.plot(ser_le_male.index, ser_le_male.values, color='lightskyblue', linestyle='-', linewidth=3, label='мужчины, ОПЖ' if lang=='ru' else 'male, LE', zorder=13)
        ax.plot(ser_hale_male.index, ser_hale_male.values, color='blue', linestyle='-', linewidth=3, label='мужчины, ОПЗЖ' if lang=='ru' else 'male, HALE', zorder=16)
    
        ax.legend(loc=chart_params.legend_loc, ncol=3, fontsize=12)

    else:
        ax.plot(ser_le_female.index, ser_le_female.values, color='lightpink', linestyle='-', linewidth=3, label='женщины, ОПЖ' if lang=='ru' else 'female, LE', zorder=12)
        ax.plot(ser_le_total.index, ser_le_total.values, color='palegreen', linestyle='-', linewidth=6, label='всё население, ОПЖ' if lang=='ru' else 'overall, LE', zorder=11)
        ax.plot(ser_le_male.index, ser_le_male.values, color='lightskyblue', linestyle='-', linewidth=3, label='мужчины, ОПЖ' if lang=='ru' else 'male, LE', zorder=13)
        ax.plot(ser_hale_female.index, ser_hale_female.values, color='red', linestyle='-', linewidth=3, label='женщины, ОПЗЖ' if lang=='ru' else 'female, HALE', zorder=15) 
        ax.plot(ser_hale_total.index, ser_hale_total.values, color='forestgreen', linestyle='-', linewidth=6, label='всё население, ОПЗЖ' if lang=='ru' else 'overall, HALE', zorder=14)
        ax.plot(ser_hale_male.index, ser_hale_male.values, color='blue', linestyle='-', linewidth=3, label='мужчины, ОПЗЖ' if lang=='ru' else 'male, HALE', zorder=16)
    
        ax.legend(loc=chart_params.legend_loc, ncol=chart_params.legend_ncol, fontsize=12)

In [14]:
# This decorator trace what countries were processed.
# Result is kept in a list "ls_log"
ls_log = []

def logger(fn):
    def inner(*args, **kwargs):
        ls_log.append(kwargs['file_name'])
        return fn(*args, **kwargs)
    return inner

In [15]:
@logger
def create_chart(ser_le_total=None, ser_le_male=None, ser_le_female=None,
                 ser_hale_total=None, ser_hale_male=None, ser_hale_female=None, *,
                 figure_size=(15, 10), title_en='', title_ru='', ver_lines=[], 
                 lang='en', chart_complexity=CHART_COMPLEXITY, chart_params=ChartParams(),
                 destination=DESTINATION_OUTPUT, file_name=''):

    if (ser_le_total.name in LS_IGNORE) or (destination == 'pass'):
        return None

    if chart_complexity == 'all_to_files':
        for lang in ['en', 'ru']:
            create_chart_simple(
                ser_le_total=ser_le_total, ser_le_male=ser_le_male, ser_le_female=ser_le_female,
                ser_hale_total=ser_hale_total, ser_hale_male=ser_hale_male, ser_hale_female=ser_hale_female,
                figure_size=figure_size, title_en=title_en, title_ru=title_ru, ver_lines=ver_lines, 
                lang=lang, chart_params=chart_params, file_name=file_name)
            process_chart_result(plt, lang=lang, destination='file', file_name=file_name)
    else:
        create_chart_simple(
            ser_le_total=ser_le_total, ser_le_male=ser_le_male, ser_le_female=ser_le_female,
            ser_hale_total=ser_hale_total, ser_hale_male=ser_hale_male, ser_hale_female=ser_hale_female,
            figure_size=figure_size, title_en=title_en, title_ru=title_ru, ver_lines=ver_lines, 
            lang=lang, chart_params=chart_params, file_name=file_name)
        process_chart_result(plt, lang=lang, destination=destination, file_name=file_name)

<br />
<br />

### post-USSR-countries

[Список союзных республик СССР](https://ru.wikipedia.org/wiki/Список_союзных_республик_СССР#Конституционный_порядок) /
[Republics of the Soviet Union](https://en.wikipedia.org/wiki/Republics_of_the_Soviet_Union#Union_Republics_of_the_Soviet_Union)

In [17]:
region = 'Russia'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Russia', title_ru='в России', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [18]:
region = 'Belarus'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Belarus', title_ru='в Белоруссии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [19]:
region = 'Ukraine'
ver_lines = [2013, 2014, 2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Ukraine', title_ru='на Украине', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [20]:
region = 'Moldova'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Moldova', title_ru='в Молдавии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [21]:
region = 'Lithuania'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Lithuania', title_ru='в Литве', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [22]:
region = 'Latvia'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Latvia', title_ru='в Латвии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [23]:
region = 'Estonia'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Estonia', title_ru='в Эстонии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [24]:
region = 'Georgia'
ver_lines = [2003, 2008, 2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Georgia', title_ru='в Грузии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [25]:
region = 'Azerbaijan'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Azerbaijan', title_ru='в Азербайджане', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [26]:
region = 'Armenia'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Armenia', title_ru='в Армении', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [27]:
region = 'Kazakhstan'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Kazakhstan', title_ru='в Казахстане', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [28]:
region = 'Turkmenistan'
ver_lines = [2019]
chart_params=ChartParams(padding_up=1.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Turkmenistan', title_ru='в Туркменистане', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [29]:
region = 'Uzbekistan'
ver_lines = [2019]
chart_params=ChartParams(padding_up=-0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Uzbekistan', title_ru='в Узбекистане', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [30]:
region = 'Kyrgyzstan'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Kyrgyzstan', title_ru='в Киргизии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [31]:
region = 'Tajikistan'
ver_lines = [2019]
chart_params=ChartParams(padding_up=-0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Tajikistan', title_ru='в Таджикистане', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />
<br />
<br />

---

[Югославия](https://ru.wikipedia.org/wiki/Югославия#Состав) /
[Yugoslavia](https://en.wikipedia.org/wiki/Yugoslavia#FPR_Yugoslavia), [Yugoslavia > new states](https://en.wikipedia.org/wiki/Yugoslavia#New_states)<br />
[Распад Югославии](https://ru.wikipedia.org/wiki/Распад_Югославии) / [Breakup of Yugoslavia](https://en.wikipedia.org/wiki/Breakup_of_Yugoslavia)<br />
[Союзная Республика Югославия](https://ru.wikipedia.org/wiki/Союзная_Республика_Югославия) / [Federal Republic of Yugoslavia](https://en.wikipedia.org/wiki/Federal_Republic_of_Yugoslavia)

In [33]:
region = 'Slovenia'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Slovenia', title_ru='в Словении', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [34]:
region = 'Croatia'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Croatia', title_ru='в Хорватии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [35]:
region = 'Bosnia and Herzegovina'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Bosnia and Herzegovina', title_ru='в Боснии и Герцеговине', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [36]:
region = 'Montenegro'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Montenegro', title_ru='в Черногории', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [37]:
region = 'Serbia'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Serbia', title_ru='в Сербии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [38]:
region = 'North Macedonia'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in North Macedonia', title_ru='в Северной Македонии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />
<br />
<br />

---

[Чехословакия](https://ru.wikipedia.org/wiki/Чехословакия) / [Czechoslovakia](https://en.wikipedia.org/wiki/Czechoslovakia)<br />
[Чешская и Словацкая Федеративная Республика](https://ru.wikipedia.org/wiki/Чешская_и_Словацкая_Федеративная_Республика) / [Czech and Slovak Federative Republic](https://en.wikipedia.org/wiki/Czech_and_Slovak_Federative_Republic)<br />
[Бархатный развод](https://ru.wikipedia.org/wiki/) / [Dissolution of Czechoslovakia](https://en.wikipedia.org/wiki/Dissolution_of_Czechoslovakia)

In [40]:
region = 'Czechia'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Czechia', title_ru='в Чехии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [41]:
region = 'Slovakia'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Slovakia', title_ru='в Словакии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />
<br />
<br />

---
### Europe

In [43]:
region = 'Iceland'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5, legend_ncol=3)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Iceland', title_ru='в Исландии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [44]:
region = 'Ireland'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Ireland', title_ru='в Ирландии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [45]:
region = 'United Kingdom'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0, legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in the United Kingdom', title_ru='в Великобритании', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [46]:
region = 'Portugal'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Portugal', title_ru='в Португалии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [47]:
region = 'Spain'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Spain', title_ru='в Испании', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [48]:
region = 'Italy'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Italy', title_ru='в Италии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [49]:
region = 'Greece'
ver_lines = [2019]
chart_params=ChartParams(padding_up=1 if LANG=='ru' else 0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Greece', title_ru='в Греции', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [50]:
region = 'Albania'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Albania', title_ru='в Албании', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [51]:
region = 'France'
ver_lines = [2019]
chart_params=ChartParams(padding_up=1)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in France', title_ru='во Франции', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [52]:
region = 'Belgium'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Belgium', title_ru='в Бельгии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)


In [53]:
region = 'Netherlands'
ver_lines = [2019]
chart_params=ChartParams(padding_up=-0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in the Netherlands', title_ru='в Нидерландах', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [54]:
region = 'Germany'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Germany', title_ru='в Германии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [55]:
region = 'Switzerland'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Switzerland', title_ru='в Швейцарии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [56]:
region = 'Austria'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Austria', title_ru='в Австрии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [57]:
region = 'Denmark'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Denmark', title_ru='в Дании', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [58]:
region = 'Norway'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Norway', title_ru='в Норвегии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [59]:
region = 'Sweden'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Sweden', title_ru='в Швеции', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [60]:
region = 'Finland'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Finland', title_ru='в Финляндии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)


In [61]:
region = 'Poland'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Poland', title_ru='в Польше', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [62]:
region = 'Romania'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Romania', title_ru='в Румынии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [63]:
region = 'Hungary'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Hungary', title_ru='в Венгрии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [64]:
region = 'Bulgaria'
ver_lines = [2019]
chart_params=ChartParams(padding_up=1 if LANG=='ru' else 0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Bulgaria', title_ru='в Болгарии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [65]:
region = 'Malta'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Malta', title_ru='на Мальте', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [66]:
region = 'Luxembourg'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Luxembourg', title_ru='в Люксембурге', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />
<br />
<br />

---
### Asia

In [68]:
region = 'Turkey'
ver_lines = [2019]
chart_params=ChartParams(padding_up=1, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Turkey', title_ru='в Турции', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [69]:
region = 'Cyprus'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Cyprus', title_ru='на Кипре', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />

[Иран](https://ru.wikipedia.org/wiki/Иран) / [Iran](https://en.wikipedia.org/wiki/Iran)

In [71]:
region = 'Iran'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Iran', title_ru='в Иране', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />

[Афганистан](https://ru.wikipedia.org/wiki/Афганистан) / [Afghanistan](https://en.wikipedia.org/wiki/Afghanistan)

In [73]:
region = 'Afghanistan'
ver_lines = [2001, 2004, 2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Afghanistan', title_ru='в Афганистане', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [74]:
region = 'Pakistan'
ver_lines = [2019]
chart_params=ChartParams(padding_up=-0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Pakistan', title_ru='в Пакистане', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [75]:
region = 'Bhutan'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Bhutan', title_ru='в Бутане', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [76]:
region = 'India'
ver_lines = [2019]
chart_params=ChartParams(padding_up=-0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in India', title_ru='в Индии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [77]:
region = 'Sri Lanka'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Sri Lanka', title_ru='в Шри-Ланке', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [78]:
region = 'Maldives'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in the Maldives', title_ru='на Мальдивах', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [79]:
region = 'Bangladesh'
ver_lines = [2019]
chart_params=ChartParams(padding_up=-0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Bangladesh', title_ru='в Бангладеше', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [80]:
region = 'Myanmar'
ver_lines = [2019]
chart_params=ChartParams(padding_up=-0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Myanmar (until 1989 Birma)', title_ru='в Мьянме', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [81]:
region = 'Thailand'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Thailand', title_ru='в Тайланде', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [82]:
region = 'Cambodia'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5, legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Cambodia', title_ru='в Камбодже', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />

[Вьетнам](https://ru.wikipedia.org/wiki/Вьетнам) / [Vietnam](https://en.wikipedia.org/wiki/Vietnam)

In [84]:
region = 'Vietnam'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Vietnam', title_ru='во Вьетнаме', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [85]:
region = 'Malaysia'
ver_lines = [2019]
chart_params=ChartParams(padding_up=-0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Malaysia', title_ru='в Малайзии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [86]:
region = 'Indonesia'
ver_lines = [2019]
chart_params=ChartParams(padding_up=-0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Indonesia', title_ru='в Индонезии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [87]:
region = 'Philippines'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5, legend_loc='lower left')
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in the Philippines', title_ru='на Филиппинах', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [88]:
region = 'Singapore'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Singapore', title_ru='в Сингапуре', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [89]:
region = 'China'
ver_lines = [2003, 2013, 2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in China', title_ru='в Китае', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [90]:
region = 'Mongolia'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Mongolia', title_ru='в Монголии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [91]:
region = 'North Korea'
ver_lines = [2019]
chart_params=ChartParams(legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in North Korea', title_ru='в Северной Корее', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [92]:
region = 'South Korea'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in South Korea', title_ru='в Южной Корее', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [93]:
region = 'Japan'
ver_lines = [2011, 2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Japan', title_ru='в Японии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [94]:
region = 'Australia'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Australia', title_ru='в Австралии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [95]:
region = 'New Zealand'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in New Zealand', title_ru='в Новой Зеландии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />

[Израиль](https://ru.wikipedia.org/wiki/Израиль) / [Israel](https://en.wikipedia.org/wiki/Israel)

In [97]:
region = 'Israel'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Israel', title_ru='в Израиле', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [98]:
# region = 'occupied Palestinian territory'
# ver_lines = [2019]
# chart_params=ChartParams()
# create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
#              df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
#              title_en='in the occupied Palestinian territory, including East Jerusalem', title_ru='\nна оккупированной палестинской территории, включая Восточный Иерусалим', ver_lines=ver_lines,
#              lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [99]:
region = 'Lebanon'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Lebanon', title_ru='в Ливане', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [100]:
region = 'United Arab Emirates'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in the United Arab Emirates', title_ru='в ОАЭ', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [101]:
region = 'Saudi Arabia'
ver_lines = [2019]
chart_params=ChartParams(padding_up=-0.5, padding_down=-0.5, legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Saudi Arabia', title_ru='в Саудовской Аравии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [102]:
region = 'Jordan'
ver_lines = [2019]
chart_params=ChartParams(padding_up=-0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Jordan', title_ru='в Иордании', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />

[Сирия](https://ru.wikipedia.org/wiki/Сирия) / [Syria](https://en.wikipedia.org/wiki/Syria)

In [104]:
region = 'Syria'
ver_lines = [2011, 2019]
chart_params=ChartParams(padding_down=-0.5, legend_loc='lower left', legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Syria', title_ru='в Сирии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />

[Ирак](https://ru.wikipedia.org/wiki/Ирак) / [Iraq](https://en.wikipedia.org/wiki/Iraq)

In [106]:
region = 'Iraq'
ver_lines = [2003, 2011, 2017, 2019, 2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Iraq', title_ru='в Ираке', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />

[Кувейт](https://ru.wikipedia.org/wiki/Кувейт) / [Kuwait](https://en.wikipedia.org/wiki/Kuwait)

In [108]:
region = 'Kuwait'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Kuwait', title_ru='в Кувейте', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [109]:
region = 'Qatar'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Qatar', title_ru='в Катаре', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [110]:
region = 'Oman'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Oman', title_ru='в Омане', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [111]:
region = 'Yemen'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5, legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Yemen', title_ru='в Йемене', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />
<br />
<br />

---
### Africa

In [113]:
region = 'Egypt'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Egypt', title_ru='в Египте', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [114]:
region = 'Seychelles'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Seychelles', title_ru='на Сейшельских Островах', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [115]:
region = 'Algeria'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Algeria', title_ru='в Алжире', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [116]:
region = 'Morocco'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Morocco', title_ru='в Марокко', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [117]:
region = 'Tunisia'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Tunisia', title_ru='в Тунисе', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [118]:
region = 'Libya'
ver_lines = [2011, 2019]
chart_params=ChartParams(padding_down=-0.5, legend_loc='lower left', legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Libya', title_ru='в Ливии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [119]:
region = 'South Africa'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in South Africa', title_ru='\nв Южно-Африканской Республике', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [120]:
region = 'Nigeria'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Nigeria', title_ru='в Нигерии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [121]:
region = 'Ethiopia'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Ethiopia', title_ru='в Эфиопии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [122]:
region = 'Kenya'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Kenya', title_ru='в Кении', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [123]:
region = 'Tanzania'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Tanzania', title_ru='в Танзании', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [124]:
region = 'Congo, DR'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in the Democratic Republic of the Cong', title_ru='в ДР Конго', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [125]:
region = 'Somalia'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Somalia', title_ru='в Сомали', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [126]:
region = 'Zimbabwe'
ver_lines = [2019]
chart_params=ChartParams(legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Zimbabwe', title_ru='в Зимбабве', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [127]:
region = 'Uganda'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Uganda', title_ru='в Уганде', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [128]:
region = 'Botswana'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Botswana', title_ru='в Ботсване', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [129]:
region = 'Angola'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Angola', title_ru='в Анголе', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [130]:
region = 'Niger'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Niger', title_ru='в Нигере', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [131]:
region = 'Mali'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Mali', title_ru='в Мали', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [132]:
region = 'Sudan'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Sudan', title_ru='в Судане', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [133]:
region = 'Zambia'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Zambia', title_ru='в Замбии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [134]:
region = 'Mozambique'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-1)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Mozambique', title_ru='в Мозамбике', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [135]:
region = 'Ghana'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Ghana', title_ru='в Гане', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [136]:
region = 'Namibia'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Namibia', title_ru='в Намибии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [137]:
region = 'Eswatini'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Eswatini', title_ru='в Эсватини', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [138]:
region = 'Malawi'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Malawi', title_ru='в Малави', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [139]:
region = 'Cameroon'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Cameroon', title_ru='в Камеруне', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [140]:
region = 'Lesotho'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Lesotho', title_ru='в Лесото', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [141]:
region = 'Madagascar'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Madagascar', title_ru='на Мадагаскаре', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [142]:
region = 'Rwanda'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Rwanda', title_ru='в Руанде', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [143]:
region = "Cote d'Ivoire"
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5, legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en="in Côte d'Ivoire", title_ru='в Кот-д’Ивуаре', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [144]:
region = 'Liberia'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Liberia', title_ru='в Либерии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [145]:
region = 'Senegal'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Senegal', title_ru='в Сенегале', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [146]:
region = 'Burkina Faso'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5, legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Burkina Faso', title_ru='в Буркина-Фасо', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [147]:
region = 'Mauritania'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Mauritania', title_ru='в Мавритании', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [148]:
region = 'Cape Verde'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Cape Verde', title_ru='в Кабо-Верде', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [149]:
region = 'Mauritius'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Mauritius', title_ru='на Маврикии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [150]:
region = 'Central African Republic'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5, legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in the Central African Republic (CAR)', title_ru='\nв Центральноафриканской Республике (ЦАР)', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [151]:
region = 'Congo Republic'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in the Republic of the Congo', title_ru='в Республике Конго', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [152]:
region = 'Togo'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Togo', title_ru='в Того', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [153]:
region = 'Chad'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Chad', title_ru='в Чаде', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />
<br />
<br />

---
### North America

In [155]:
region = 'USA'
ver_lines = [2001, 2005, 2019]
chart_params=ChartParams(padding_up=1, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in the United States', title_ru='в США', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [156]:
region = 'Canada'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Canada', title_ru='в Канаде', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [157]:
region = 'Mexico'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Mexico', title_ru='в Мексике', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [158]:
region = 'Guatemala'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Guatemala', title_ru='в Гватемале', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [159]:
region = 'Honduras'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Honduras', title_ru='в Гондурасе', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [160]:
region = 'El Salvador'
ver_lines = [2019]
chart_params=ChartParams(padding_up=1, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in El Salvador', title_ru='в Эль-Сальвадоре', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [161]:
region = 'Nicaragua'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Nicaragua', title_ru='в Никарагуа', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [162]:
region = 'Costa Rica'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Costa Rica', title_ru='в Коста-Рике', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [163]:
region = 'Panama'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Panama', title_ru='в Панаме', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />

[Куба](https://ru.wikipedia.org/wiki/Куба) / [Cuba](https://en.wikipedia.org/wiki/Cuba)

In [165]:
region = 'Cuba'
ver_lines = [2008, 2019]
chart_params=ChartParams(padding_down=-0.5, legend_loc='lower left', legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Cuba', title_ru='на Кубе', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [166]:
region = 'Haiti'
ver_lines = [2010, 2019]
chart_params=ChartParams(padding_up=1, padding_down=-0.5, legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in the Republic of Haiti', title_ru='в Республике Гаити', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [167]:
region = 'Dominican Republic'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in the Dominican Republic', title_ru='\nв Доминиканской Республике', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [168]:
region = 'Trinidad and Tobago'
ver_lines = [2019]
chart_params=ChartParams(padding_up=1 if LANG=='ru' else 0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Trinidad and Tobago', title_ru='в Тринидад и Тобаго', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [169]:
region = 'Jamaica'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Jamaica', title_ru='на Ямайке', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [170]:
region = 'Puerto Rico'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Puerto Rico', title_ru='в Пуэрто-Рико', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />
<br />
<br />

---
### South America

In [172]:
region = 'Chile'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Chile', title_ru='в Чили', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [173]:
region = 'Uruguay'
ver_lines = [2019]
chart_params=ChartParams(padding_up=1)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Uruguay', title_ru='в Уругвае', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [174]:
region = 'Colombia'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Colombia', title_ru='в Колумбии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [175]:
region = 'Ecuador'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Ecuador', title_ru='в Эквадоре', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [176]:
region = 'Peru'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Peru', title_ru='в Перу', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [177]:
region = 'Argentina'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Argentina', title_ru='в Аргентине', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [178]:
region = 'Brazil'
ver_lines = [2019]
chart_params=ChartParams(padding_up=0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Brazil', title_ru='в Бразилии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [179]:
region = 'Venezuela'
ver_lines = [2019]
chart_params=ChartParams(legend_loc='lower left', legend_ncol=2)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Venezuela', title_ru='в Венесуэле', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [180]:
region = 'Bolivia'
ver_lines = [2019]
chart_params=ChartParams(padding_up=-0.5, padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Bolivia', title_ru='в Боливии', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [181]:
region = 'Paraguay'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-1, legend_loc='lower left')
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Paraguay', title_ru='в Парагвае', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />
<br />
<br />

In [183]:
region = 'world'
ver_lines = [2019]
chart_params=ChartParams(padding_down=-0.5)
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in the world', title_ru='в мире', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [184]:
region = 'Africa'
ver_lines = [2019]
chart_params=ChartParams()
create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
             df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
             title_en='in Africa', title_ru='в Африке', ver_lines=ver_lines,
             lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [185]:
# region = 'Europe'
# ver_lines = [2019]
# chart_params=ChartParams()
# create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
#              df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
#              title_en='in Europe', title_ru='в Европе', ver_lines=ver_lines,
#              lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [186]:
# region = 'High-income'
# ver_lines = [2019]
# chart_params=ChartParams()
# create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
#              df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
#              title_en='in the high-income countries', title_ru='в странах с высоким уровнем дохода', ver_lines=ver_lines,
#              lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

In [187]:
# region = 'Low-income'
# ver_lines = [2019]
# chart_params=ChartParams()
# create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
#              df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
#              title_en='in the low-income countries', title_ru='в странах с низким уровнем дохода', ver_lines=ver_lines,
#              lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

<br />
<br />

<h4>Comparison of countries with various income:</h4>

In [189]:
TitlePars  = namedtuple('TitlePars', ['label_en', 'label_ru', 'fontsize'], defaults=('', '', 14))

dd_to_rus = {'High-income': 'Высокий доход',
             'Upper-middle-income': 'Средне-высокий доход',
             'Lower-middle-income': 'Средне-низкий доход',
             'Low-income': 'Низкий доход'}

In [190]:
# This function was created quickly, the code can be not optimal
def create_chart_group(df_le, df_hale, colors_le, colors_hale, *,
                       figure_size=(15, 10), title=TitlePars(), legend='',
                       chart_lim=None, linewidth=3, lang=LANG, chart_params=ChartParams(),
                       xticks_size=13, yticks_size=11.5, destination=DESTINATION_OUTPUT, file_name=''):
    if destination == 'pass':
        return None

    plt.rcParams['figure.figsize'] = figure_size

    # determine range of years
    year_min = df_le.columns[0]
    year_max = df_le.columns[-1]
    print(year_min, year_max)

    # determine limits on axis
    min_chart = df_hale.min().min()
    max_chart = df_le.max().max()
    # print(f"min: {min_chart:.2f}, max: {max_chart:.2f}")
    
    ylim_min = rounding_down(min_chart) - chart_params.padding_down
    ylim_max = rounding_up(max_chart)   + chart_params.padding_up
    # print(f"min_corrected: {ylim_min:.2f}, max_corrected: {ylim_max:.2f}")

    min_for_labels = math.ceil(ylim_min)
    max_for_labels = math.floor(ylim_max)
    print(f"min_for_labels / max_for_labels: {min_for_labels} - {max_for_labels}")

    # determine which label have to be not shown
    empty_labels_down = (round(min_chart) - min_for_labels) + chart_params.add_empty_labels_down
    empty_labels_up   = max_for_labels - round(max_chart) + chart_params.add_empty_labels_up

    # set chart areas, their locations, sizes and appearances
    fig = plt.figure()
    ax = fig.add_subplot(111)


    # set where ticks and labels will be shown around charts
    ax.tick_params(axis='x', which='both', top=False, bottom=True, labeltop=False, labelbottom=True, pad=1.5)
    ax.tick_params(axis='y', which='both', left=True, right=True, labelleft=True, labelright=True, pad=1.5)

    # set limits of chart axes
    ax.set_xlim(year_min, year_max)
    ax.set_ylim(ylim_min, ylim_max)


    # set ticks and labels
    labels_x = range(year_min, year_max+1)
    ax.set_xticks(labels_x)
    ax.set_xticklabels(labels_x, fontsize=12, rotation=chart_params.label_x_rotation)

    labels_y = tuple(range(min_for_labels, max_for_labels + 1))
    ax.set_yticks(labels_y)
    ax.set_yticklabels(('',) * empty_labels_down + 
                        labels_y[empty_labels_down:-empty_labels_up if empty_labels_up else None] +
                        ('',) * empty_labels_up, fontsize=11)

    # settings grids
    ax.grid(color='wheat', linewidth=0.25)

    for year in range(year_min - year_min % 10 + 10, year_max, 10):
        ax.axvline(x=year, color='lightgrey', linewidth=0.4, zorder=0)

    for age in range(min_for_labels - min_for_labels % 5 + 5, max_for_labels, 5):
        ax.axhline(y=age, color='lightgrey', linewidth=0.4, zorder=0)

    for year in ver_lines:
        ax.axvline(x=year, color='wheat', linewidth=2, alpha=0.5, zorder=0)

    # set title
    ax.set_title(label=title.label_ru if lang=='ru' else title.label_en, fontsize=title.fontsize)

    for z, country, color in zip(range(len(df_hale),0,-1), df_hale.loc[ls_countries].index, colors_hale):
        label = "ОПЗЖ: " + dd_to_rus[country] if lang=='ru' else "HALE: " + country.replace('-', ' ')
        plt.plot(df_hale.columns, df_hale.loc[country], linestyle='-', color=color, linewidth=linewidth, label=label, zorder=z)

    for z, country, color in zip(range(len(df_le),0,-1), df_le.loc[ls_countries].index, colors_le):
        label = "ОПЖ: " + dd_to_rus[country] if lang=='ru' else "LE: " + country.replace('-', ' ')
        plt.plot(df_le.columns, df_le.loc[country], linestyle='-', color=color, linewidth=linewidth, label=label, zorder=50+z, alpha=0.75)

    ax.legend(loc=chart_params.legend_loc, ncol=chart_params.legend_ncol, fontsize=10.4 if lang=='ru' else 11.6)

    if destination == 'file':
        plt.savefig(f"output-charts-groups/HALE and Life Expectancy by WHO -{file_name}{' -ru' if lang=='ru' else ''}.png", bbox_inches='tight', facecolor='white', pad_inches=0.05)
        print('Data has written to file')
    else:
        plt.show()



title = TitlePars(label_en='Average Life Expectancy and Healthy Life Expectancy (HALE) in groups of countries with different income',
                  label_ru='Средние ожидаемая продолжительность жизни и ожидаемая продолжительность здоровой жизни\n' + \
                           'в группах стран с различным уровнем дохода',
                  fontsize=13)

chart_params=ChartParams()

ls_countries = ['High-income', 'Upper-middle-income', 'Lower-middle-income', 'Low-income']
# colors_le = ['#ea580c', '#f97316', '#fb923c', '#fdba74']    # orange palette
# colors_hale = ['#4d7c0f', '#65a30d', '#84cc16', '#a3e635']  # lime palette
# colors_le = ['#a16207', '#ca8a04', '#eab308', '#facc15']    # yellow palette
# colors_hale = ['#1e40af', '#2563eb', '#60a5fa', '#93c5fd']  # blue palette
colors_hale = ['#004800', 'green', 'limegreen', 'lime']
colors_le = ['chocolate', 'darkorange', 'orange', 'gold']

chart_params=ChartParams(padding_up=2.5, padding_down=-1, legend_ncol=4)
create_chart_group(df_le_total.loc[ls_countries], df_hale_total.loc[ls_countries], colors_le, colors_hale,
                   title=title, lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT,
                   file_name='income comparison')

<br />

In [192]:
# play beep to denote completion of the program
import IPython.display as ipd
import numpy as np

# manually generated sound
t = 1  # time is seconds
beep = np.sin(2*np.pi*400*np.arange(10000*t)/10000)
ipd.Audio(beep, rate=10000, autoplay=True)

<br />

In [194]:
# region = '___'
# ver_lines = [2019]
# chart_params=ChartParams()
# create_chart(df_le_total.loc[region], df_le_male.loc[region], df_le_female.loc[region],
#              df_hale_total.loc[region], df_hale_male.loc[region], df_hale_female.loc[region],
#              title_en='in ___', title_ru='в ___', ver_lines=ver_lines,
#              lang=LANG, chart_params=chart_params, destination=DESTINATION_OUTPUT, file_name=region)

# padding_up=0.5, padding_down=-0.5, legend_loc='lower left'

In [195]:
print(len(ls_log))

ls_log = list(set(ls_log))
print(len(ls_log))
print(ls_log)

151
151
['Estonia', 'Montenegro', 'Ecuador', 'Costa Rica', 'Argentina', 'China', 'Algeria', 'United Kingdom', 'Romania', 'Iceland', 'Mauritius', 'Czechia', 'Cuba', 'Norway', 'South Korea', 'Puerto Rico', 'Tajikistan', 'Greece', 'Venezuela', 'Bhutan', 'Togo', 'Philippines', 'Uzbekistan', 'Dominican Republic', 'Pakistan', 'Denmark', 'Honduras', 'Spain', 'Portugal', 'Ireland', 'Mozambique', 'Serbia', 'Kazakhstan', 'Bangladesh', 'Israel', 'Chad', 'India', 'Cambodia', 'Angola', 'Panama', 'Malta', 'Nicaragua', 'Azerbaijan', 'Armenia', 'Afghanistan', 'Chile', 'Netherlands', 'Burkina Faso', 'Guatemala', 'Saudi Arabia', 'Ukraine', 'Morocco', 'Uruguay', 'Madagascar', 'Bolivia', 'Lithuania', 'Congo Republic', 'Bosnia and Herzegovina', 'Italy', 'Slovenia', 'Croatia', 'Luxembourg', 'Haiti', 'world', 'Trinidad and Tobago', 'Rwanda', 'Lesotho', 'Yemen', 'Iraq', 'El Salvador', 'Jordan', 'Ethiopia', 'Egypt', 'Eswatini', 'Peru', 'Vietnam', 'Brazil', 'Germany', 'Paraguay', 'Canada', 'Zimbabwe', 'New Zeal

countries with highlighted years:<br>
Ukraine, Georgia, Afghanistan, China, Japan, Syria, Iraq, Libya, USA, Cuba, Haiti

<br>
<br>

my small exploration (temp):

In [198]:
def analize_dataFrame(df_input, record_mean_name, sort_column=None):
    df_mean = df_input.mean().to_frame().T
    df_mean.index = [record_mean_name]
    df = pd.concat([df_mean, df_input])  # .sort_values(by='Δ1', ascending=False)

    # df.insert(loc=0,  column='mean 2000-02', value=df.loc[:, 2000:2002].mean(axis='columns'))
    # df.insert(loc=1,  column='mean 2017-19', value=df.loc[:, 2017:2019].mean(axis='columns'))
    # df.insert(loc=2,  column='mean 2020-21', value=df.loc[:, 2020:2021].mean(axis='columns'))
    # df.insert(loc=1,  column='Δ1', value=df['mean 2017-19'] - df['mean 2000-02'])
    # df.insert(loc=3,  column='Δ2', value=df['mean 2020-21'] - df['mean 2017-19'])
    df.insert(loc=0,  column='2000_', value=df[2000])
    df.insert(loc=1,  column='2019_', value=df[2019])
    df.insert(loc=2,  column='2021_', value=df[2021])
    df.insert(loc=1,  column='Δ1', value=df[2019] - df[2000])
    df.insert(loc=3,  column='Δ2', value=df[2021] - df[2019])
    df.insert(loc=5,  column='|', value='|')
    df.insert(loc=26,  column=' |', value='|')

    if sort_column:
        df.sort_values(by=sort_column, ascending=False, inplace=True)

    df = pd.concat([df.loc[[record_mean_name]], df.drop(index=record_mean_name)])

    return df

df_ratio_total = analize_dataFrame(df_hale_total / df_le_total * 100, '_mean_total_', 'Δ1')

ls_selected_countries = [
    '_mean_total_', 'France', 'Belgium', 'Switzerland', 'Spain', 'Germany', 'Italy',
    'United Kingdom', 'Netherlands', 'Sweden', 'Norway', 'Poland', 'Czechia',
    'Russia', 'Ukraine', 'Belarus', 'Kazakhstan', 'Georgia', 'Armenia', 'Azerbaijan',
    'Japan', 'South Korea', 'China', 'India', 'Thailand', 'Singapore', 'Indonesia',
    'Turkey', 'Iran', 'Israel', 'United Arab Emirates', 'Saudi Arabia',
    'South Africa',
    'Australia', 'New Zealand',
    'USA', 'Canada', 'Mexico', 'Chile', 'Argentina', 'Brazil', 'Uruguay']

df_ratio_total.loc[ls_selected_countries].sort_values(by='Δ1', ascending=False)

,2000_,Δ1,2019_,Δ2,2021_,|,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,|,2020,2021
India,85.65,0.43,86.07,0.32,86.39,|,85.65,85.64,85.65,85.69,85.67,85.67,85.70,85.79,85.83,85.80,85.82,85.83,85.84,85.89,85.91,85.89,85.91,85.98,86.06,86.07,|,86.00,86.39
Brazil,85.20,0.29,85.49,-0.08,85.41,|,85.20,85.15,85.09,85.05,85.02,84.94,85.00,85.09,85.24,85.35,85.43,85.45,85.46,85.50,85.50,85.54,85.57,85.53,85.50,85.49,|,85.37,85.41
Indonesia,88.42,0.22,88.64,0.24,88.88,|,88.42,88.46,88.51,88.56,88.71,88.63,88.64,88.67,88.68,88.71,88.72,88.74,88.72,88.71,88.71,88.70,88.68,88.66,88.65,88.64,|,88.87,88.88
Russia,87.03,-0.01,87.03,0.01,87.03,|,87.03,87.08,87.20,87.31,87.30,87.36,87.23,87.17,87.20,87.15,87.16,87.09,87.08,87.09,87.18,87.17,87.17,87.10,87.10,87.03,|,87.17,87.03
Mexico,86.36,-0.04,86.32,0.42,86.74,|,86.36,86.34,86.35,86.36,86.30,86.33,86.25,86.20,86.24,86.20,86.26,86.17,86.17,86.17,86.22,86.25,86.30,86.29,86.29,86.32,|,86.81,86.74
Switzerland,85.75,-0.06,85.69,-0.30,85.38,|,85.75,85.73,85.77,85.90,85.87,85.90,85.82,85.77,85.70,85.64,85.60,85.59,85.64,85.65,85.60,85.68,85.61,85.66,85.68,85.69,|,85.65,85.38
Belarus,87.53,-0.06,87.46,-0.33,87.13,|,87.53,87.60,87.70,87.67,87.65,87.69,87.62,87.54,87.57,87.59,87.65,87.63,87.49,87.48,87.45,87.43,87.45,87.47,87.51,87.46,|,87.52,87.13
Armenia,87.92,-0.07,87.86,-0.15,87.71,|,87.92,87.92,88.08,88.12,88.12,88.15,88.18,88.14,88.15,88.15,88.12,88.08,88.05,88.00,88.02,88.00,88.00,87.94,87.83,87.86,|,88.03,87.71
Ukraine,87.03,-0.09,86.95,-0.15,86.80,|,87.03,87.03,87.15,87.28,87.36,87.48,87.43,87.47,87.45,87.29,87.24,87.14,87.14,87.11,87.19,87.04,86.98,86.93,86.98,86.95,|,86.88,86.80
Italy,86.17,-0.10,86.07,-0.23,85.84,|,86.17,86.12,86.09,86.17,86.05,86.11,86.04,86.08,86.11,86.15,86.12,86.18,86.22,86.17,86.15,86.25,86.12,86.19,86.10,86.07,|,86.01,85.84


Countries with best and worst dynamics if we consider years 2000, 2019, 2021:<br>

+: Nigeria, Madagascar, India, Cape Verde, Brazil, Indonesia | Russia, Mexico, Switzerland, Ukraine, Italy, Japan, _ Poland, Turkey, North Korea, Norway<br>
-: New Zealand, Israel, Singapore, Australia, France, Iran, Czechia, Uruguay, _ United Kingdom, China, Germany, Belgium, Chile, Sweden, United Arab Emirates, Saudi Arabia, Spain, Canada, Netherlands, South Africa, South Korea, USA

<br>

Countries with best and worst dynamics if we consider means for 2000-02, 2017-19, 2020-21:

+: Madagascar, Nigeria, India, Cape Verde, Brazil, Indonesia | Italy, Russia, Mexico, Japan, Switzerland, _ Turkey, Poland, Ukraine, Norway, North Korea<br>
-: Israel, New Zealand, Australia, France, Czechia, United Kingdom, Uruguay, Germany, China, Belgium, Chile, Sweden, Spain, Canada, Netherlands, South Korea, South Africa, USA

In [201]:
# some exact values
print(df_ratio_total.loc['_mean_total_', 2000])
print(df_ratio_total.loc['_mean_total_', 2019] - df_ratio_total.loc['_mean_total_', 2000])
print(df_ratio_total.loc['_mean_total_', 2019])
print(df_ratio_total.loc['_mean_total_', 2021] - df_ratio_total.loc['_mean_total_', 2019])
print(df_ratio_total.loc['_mean_total_', 2021])    

87.11867806623243
-0.19124149349056552
86.92743657274187
-0.09646996144755349
86.83096661129431


<br>

In [203]:
pd.options.display.min_rows = 20
pd.options.display.max_rows = 100

In [204]:
df_ratio_total  # .iloc[100:160]   # df_ratio_total.sort_values(by="2019_", ascending=False)

,2000_,Δ1,2019_,Δ2,2021_,|,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,|,2020,2021
_mean_total_,87.12,-0.19,86.93,-0.10,86.83,|,87.12,87.10,87.11,87.12,87.11,87.10,87.07,87.06,87.05,87.04,87.04,87.01,87.00,86.99,86.99,87.00,86.98,86.96,86.94,86.93,|,86.87,86.83
Timor-Leste,85.86,1.69,87.55,-0.00,87.54,|,85.86,85.97,86.25,86.59,86.90,87.04,87.10,87.17,87.22,87.30,87.36,87.44,87.53,87.65,87.76,87.77,87.73,87.65,87.58,87.55,|,87.49,87.54
Mozambique,85.48,1.09,86.57,-0.34,86.23,|,85.48,85.52,85.70,85.93,86.11,86.23,86.31,86.37,86.47,86.54,86.61,86.64,86.70,86.74,86.75,86.73,86.70,86.66,86.61,86.57,|,86.56,86.23
"Congo, DR",85.54,0.97,86.51,-0.18,86.34,|,85.54,85.53,85.52,85.53,85.52,85.49,85.56,85.71,85.92,86.08,86.18,86.23,86.30,86.37,86.41,86.44,86.45,86.47,86.49,86.51,|,86.33,86.34
Burkina Faso,86.84,0.94,87.78,-0.21,87.58,|,86.84,86.90,87.00,87.10,87.21,87.25,87.26,87.29,87.31,87.34,87.39,87.47,87.57,87.68,87.76,87.79,87.79,87.82,87.80,87.78,|,87.64,87.58
Maldives,87.93,0.78,88.71,-0.29,88.42,|,87.93,87.99,88.12,88.24,88.34,88.32,88.35,88.37,88.39,88.41,88.41,88.40,88.25,88.22,88.14,88.08,88.01,87.94,87.86,88.71,|,88.44,88.42
Philippines,87.54,0.70,88.23,0.26,88.50,|,87.54,87.65,87.78,87.83,87.96,87.99,88.00,87.97,88.01,88.04,88.02,88.03,88.12,88.14,88.12,88.20,88.22,88.15,88.16,88.23,|,88.02,88.50
Nigeria,86.26,0.65,86.90,-0.23,86.67,|,86.26,86.30,86.35,86.42,86.46,86.45,86.48,86.54,86.62,86.69,86.74,86.76,86.79,86.81,86.82,86.83,86.85,86.87,86.89,86.90,|,86.79,86.67
Sierra Leone,86.50,0.63,87.14,-0.07,87.07,|,86.50,86.50,86.54,86.55,86.58,86.61,86.68,86.82,86.98,87.12,87.16,87.19,87.17,87.19,87.36,87.27,87.20,87.19,87.14,87.14,|,87.00,87.07
Madagascar,87.40,0.60,88.00,-0.38,87.62,|,87.40,87.41,87.48,87.53,87.56,87.56,87.58,87.64,87.75,87.84,87.91,87.94,87.98,87.99,88.03,88.05,88.04,88.03,88.02,88.00,|,87.90,87.62


<br>

In [206]:
# check whether there are countries where ratio HALE / LE for female is highter than for males
df_ratio_male   = analize_dataFrame(df_hale_male   / df_le_male   * 100, '_mean_male_'  , 'Δ1')
df_ratio_female = analize_dataFrame(df_hale_female / df_le_female * 100, '_mean_female_', 'Δ1')

In [207]:
year = 2019
t = pd.concat([df_ratio_female[year].sort_index(),
               df_ratio_total[year].sort_index(),
               df_ratio_male[year].sort_index()], axis='columns')
t.columns = ['female', 'total', 'male']
t.loc[t['female'] >= t['male']]

,female,total,male


In [208]:
for year in range(2000, 2022):
    t = pd.concat([df_ratio_female[year].sort_index(),
               df_ratio_total[year].sort_index(),
               df_ratio_male[year].sort_index()], axis='columns')
    t.columns = ['female', 'total', 'male']
    print(f"{year}: {len(t.loc[t['female'] >= t['male']])}")

2000: 0
2001: 0
2002: 0
2003: 0
2004: 0
2005: 0
2006: 0
2007: 0
2008: 0
2009: 0
2010: 0
2011: 0
2012: 0
2013: 0
2014: 0
2015: 0
2016: 0
2017: 0
2018: 0
2019: 0
2020: 0
2021: 0


<br>

In [210]:
# mean values for all countries (that can be approximately interpreted as world)
df_means = pd.concat([analize_dataFrame(df_hale_male   / df_le_male * 100  , '_mean_male_')  .loc[['_mean_male_']],
                      analize_dataFrame(df_hale_total  / df_le_total * 100 , '_mean_total_') .loc[['_mean_total_']],
                      analize_dataFrame(df_hale_female / df_le_female * 100, '_mean_female_').loc[['_mean_female_']],
                     ])
df_means

,2000_,Δ1,2019_,Δ2,2021_,|,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,|,2020,2021
_mean_male_,88.48,-0.12,88.36,0.01,88.37,|,88.48,88.46,88.47,88.49,88.48,88.48,88.45,88.45,88.46,88.45,88.44,88.42,88.41,88.41,88.41,88.41,88.40,88.39,88.38,88.36,|,88.40,88.37
_mean_total_,87.12,-0.19,86.93,-0.10,86.83,|,87.12,87.10,87.11,87.12,87.11,87.10,87.07,87.06,87.05,87.04,87.04,87.01,87.00,86.99,86.99,87.00,86.98,86.96,86.94,86.93,|,86.87,86.83
_mean_female_,85.80,-0.26,85.53,-0.21,85.32,|,85.80,85.78,85.79,85.79,85.77,85.75,85.72,85.70,85.69,85.67,85.67,85.64,85.63,85.61,85.61,85.61,85.59,85.57,85.54,85.53,|,85.37,85.32


In [211]:
# chart for mean values
# ver_lines = [2019]
# chart_params=ChartParams(padding_down=-0.5)
# create_chart(df_le_total.mean(), df_le_male.mean(), df_le_female.mean(),
#              df_hale_total.mean(), df_hale_male.mean(), df_hale_female.mean(),
#              title_en=', mean for all countries', title_ru=',\nсредние показатели для всех стран', ver_lines=ver_lines,
#              lang=LANG, chart_params=chart_params, destination='show', file_name='')

<br />
<br />

<hr>
Just for fun and double check of charts: exploration of a single region in a single year:

In [213]:
year = 2019
region = 'world'

def show_statistics_for_region(df_le_total, df_le_male, df_le_female,
                               df_hale_total, df_hale_male, df_hale_female, region, year, precision=1):
    le_total,  hale_total  = df_le_total.loc[region, year],  df_hale_total.loc[region, year]
    le_male,   hale_male   = df_le_male.loc[region, year],   df_hale_male.loc[region, year]
    le_female, hale_female = df_le_female.loc[region, year], df_hale_female.loc[region, year]

    
    # ratio_total  = round(100 * le_total  / hale_total,  precision)
    # ratio_male   = round(100 * le_male   / hale_male,   precision)
    # ratio_female = round(100 * le_female / hale_female, precision)
    ratio_total  = 100 * hale_total  / le_total
    ratio_male   = 100 * hale_male   / le_male
    ratio_female = 100 * hale_female / le_female
    print(f"male  :  {hale_male:.2f} / {le_male:.2f}   →  {ratio_male:.{precision}f}%")
    print(f"total :  {hale_total:.2f} / {le_total:.2f}   →  {ratio_total:.{precision}f}%")
    print(f"female:  {hale_female:.2f} / {le_female:.2f}   →  {ratio_female:.{precision}f}%")
    return ratio_total
    
show_statistics_for_region(df_le_total, df_le_male, df_le_female, df_hale_total, df_hale_male, df_hale_female, region, year)

male  :  62.33 / 70.61   →  88.3%
total :  63.45 / 73.12   →  86.8%
female:  64.59 / 75.70   →  85.3%


86.77516411378555

<br />
<br />
<br />

generation of code for wiki-page:

In [215]:
ls_Europe = ['Albania', 'Austria', 'Belarus', 'Belgium', 'Bosnia and Herzegovina', 'Bulgaria', 'Croatia', 'Czechia', 'Denmark', 'Estonia', 'Faroe Islands', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Latvia', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Malta', 'Moldova', 'Montenegro', 'Netherlands', 'North Macedonia', 'Norway', 'Poland', 'Portugal', 'Romania', 'Russia', 'Serbia', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Ukraine', 'United Kingdom']
len(ls_Europe)

41

In [216]:
ls_Asia = ['Afghanistan', 'Armenia', 'Azerbaijan', 'Bangladesh', 'Bhutan', 'Cambodia', 'China', 'Cyprus', 'Georgia', 'Hong Kong SAR', 'India', 'Indonesia', 'Iran', 'Iraq', 'Israel', 'Japan', 'Jordan', 'Kazakhstan', 'Kuwait', 'Kyrgyzstan', 'Lebanon', 'Macao SAR', 'Malaysia', 'Maldives', 'Mongolia', 'Myanmar', 'Nepal', 'North Korea', 'Oman', 'Pakistan', 'Philippines', 'Qatar', 'Saudi Arabia', 'Singapore', 'South Korea', 'Sri Lanka', 'Syria', 'Tajikistan', 'Thailand', 'Turkey', 'Turkmenistan', 'United Arab Emirates', 'Uzbekistan', 'Vietnam', 'West Bank and Gaza', 'Yemen']
len(ls_Asia)

46

In [217]:
ls_Oceania = ['Australia', 'French Polynesia', 'New Caledonia', 'New Zealand']
len(ls_Oceania)

4

In [218]:
ls_America = ['Argentina', 'Bermuda', 'Bolivia', 'Brazil', 'British Virgin Islands', 'Canada', 'Chile', 'Colombia', 'Costa Rica', 'Cuba', 'Dominican Republic', 'Ecuador', 'El Salvador', 'Guatemala', 'Haiti', 'Honduras', 'Jamaica', 'Mexico', 'Nicaragua', 'Panama', 'Paraguay', 'Peru', 'Puerto Rico', 'Saint-Martin (France)', 'Trinidad and Tobago', 'US Virgin Islands', 'USA', 'Uruguay', 'Venezuela']
len(ls_America)

29

In [219]:
ls_Africa = ['Algeria', 'Angola', 'Botswana', 'Burkina Faso', 'Cameroon', 'Cape Verde', 'Central African Republic', 'Chad', 'Congo Republic', 'Congo, DR', "Cote d'Ivoire", 'Egypt', 'Eswatini', 'Ethiopia', 'Ghana', 'Kenya', 'Lesotho', 'Liberia', 'Libya', 'Madagascar', 'Malawi', 'Mali', 'Mauritania', 'Mauritius', 'Morocco', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 'Rwanda', 'Senegal', 'Seychelles', 'Somalia', 'South Africa', 'Sudan', 'Tanzania', 'Togo', 'Tunisia', 'Uganda', 'Zambia', 'Zimbabwe']
len(ls_Africa)

41

In [220]:
ls_combined = ls_Europe + ls_Asia + ls_Oceania + ls_America + ls_Africa
len(ls_combined)

161

In [221]:
ls_diff1 = [country for country in ls_combined if country not in ls_log]
ls_diff1

['Faroe Islands',
 'Liechtenstein',
 'Hong Kong SAR',
 'Macao SAR',
 'Nepal',
 'West Bank and Gaza',
 'French Polynesia',
 'New Caledonia',
 'Bermuda',
 'British Virgin Islands',
 'Saint-Martin (France)',
 'US Virgin Islands']

In [222]:
ls_diff2 = [country for country in ls_log if country not in ls_combined]
ls_diff2

['world', 'Africa']

In [223]:
def create_code_for_wiki(ls: list):
    ls_intersection = sorted(list(set(ls) & set(ls_log)))
    print(len(ls_intersection))
    
    for country in ls_intersection:
        print(f"[[:File:HALE and Life Expectancy by WHO -{country}.png|{country}]],")


create_code_for_wiki(ls_Oceania)

2
[[:File:HALE and Life Expectancy by WHO -Australia.png|Australia]],
[[:File:HALE and Life Expectancy by WHO -New Zealand.png|New Zealand]],
